# Model B v3 — Temporal extraction fixed to the last 2.0 seconds

Fixes the sampling bug from Week 3-4: frames were previously spread across each pedestrian's *entire* visible track, not the last 2.0 seconds before the crossing decision as the thesis plan specified.

**What changed:** `_sample_indices` now restricts the candidate pool to the final 60 frames (2.0s at 30 FPS, or the whole track if shorter), then samples 16 frames evenly within that window. T_FRAMES stays 16 per the earlier "stick with the normal plan" decision.

**New output files** (`train_temporal_crops_v3.h5` etc): the old `_crops.h5` files and all prior Model B results stay untouched for comparison.

**Open caveat for the thesis limitations section:** whether PIE's track end-frame is exactly the crossing moment or extends slightly past it hasn't been independently verified against the source code. This affects whether this is genuinely early *prediction* or partial *recognition*. Worth checking before the final write-up; not blocking this fix.

**Resumable, same as before:** a `completed` flag array lets this survive disconnects, flushes every 20 samples, and shows a live per-sample progress bar. Run this on a **CPU-only runtime** (no GPU needed for extraction, keeps your T4 quota free for training).

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PIE_thesis'
PIE_DATA_ROOT = f'{DRIVE_ROOT}/PIE_dataset'
CROPS_DIR = f'{DRIVE_ROOT}/crops'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
CHECKPOINTS_DIR = f'{DRIVE_ROOT}/checkpoints'
for d in [RESULTS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists('/content/PIE_repo'):
    get_ipython().system('git clone https://github.com/aras62/PIE.git /content/PIE_repo')
if '/content/PIE_repo/utilities' not in sys.path:
    sys.path.append('/content/PIE_repo/utilities')

import torch
import numpy as np
import h5py
import cv2
import re
from tqdm.notebook import tqdm

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', device)


Mounted at /content/drive
Cloning into '/content/PIE_repo'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 181 (delta 35), reused 75 (delta 18), pack-reused 85 (from 1)
Receiving objects: 100% (181/181), 144.63 MiB | 28.09 MiB/s, done.
Resolving deltas: 100% (76/76), done.
Updating files: 100% (41/41), done.
Device: cuda


## 2. Load PIE sequences (reuses the cached database, fast)

In [2]:
from pie_data import PIE

imdb = PIE(data_path=PIE_DATA_ROOT)
data_opts = {'seq_type': 'crossing', 'data_split_type': 'default'}

train_seqs = imdb.generate_data_trajectory_sequence('train', **data_opts)
val_seqs = imdb.generate_data_trajectory_sequence('val', **data_opts)
test_seqs = imdb.generate_data_trajectory_sequence('test', **data_opts)

print('Sequences:', len(train_seqs['pid']), len(val_seqs['pid']), len(test_seqs['pid']))

CROSS_KEY = 'activities'
IRRELEVANT_VALUE = -1
clips_root = f'{PIE_DATA_ROOT}/PIE_clips'


---------------------------------------------------------
Generating trajectory sequence data
fstride: 1
sample_type: all
height_rng: [0, inf]
squarify_ratio: 0
data_split_type: default
seq_type: crossing
min_track_size: 15
random_params: {'ratios': None, 'val_data': True, 'regen_data': False}
kfold_params: {'num_folds': 5, 'fold': 1}
---------------------------------------------------------
Generating database for pie
pie annotations loaded from /content/drive/MyDrive/PIE_thesis/PIE_dataset/data_cache/pie_database.pkl
---------------------------------------------------------
Generating crossing data
Excluding tracks with less than 15
Subset: train
Number of pedestrians: 880 
Total number of samples: 876 
---------------------------------------------------------
Generating trajectory sequence data
fstride: 1
sample_type: all
height_rng: [0, inf]
squarify_ratio: 0
data_split_type: default
seq_type: crossing
min_track_size: 15
random_params: {'ratios': None, 'val_data': True, 'regen_data

## 3. Extract T-frame crops from the LAST 2.0 seconds only (the actual fix)

`_sample_indices` now takes a `fps` and `window_seconds` argument: it first restricts to the last `window_seconds * fps` frames of the track, then samples `t_frames` evenly within that restricted window.

In [3]:
IMG_SIZE = 224
T_FRAMES = 16
FPS = 30
WINDOW_SECONDS = 2.0

def _resolve_video_path(image_ref):
    parts = image_ref.replace('\\', '/').split('/')
    set_id = next((p for p in parts if re.match(r'set\d+', p)), None)
    video_id = next((p for p in parts if re.match(r'video_\d+', p)), None)
    frame_num = int(re.sub(r'\D', '', parts[-1]))
    video_path = f'{clips_root}/{set_id}/{video_id}.mp4'
    return video_path, frame_num

def _unwrap_label(label):
    while isinstance(label, list):
        label = label[-1]
    return label

def _sample_indices(n_available, t_frames, fps, window_seconds):
    window_frames = min(n_available, int(round(fps * window_seconds)))
    start = n_available - window_frames
    if window_frames <= t_frames:
        return list(range(start, n_available))
    local_idxs = np.linspace(0, window_frames - 1, t_frames).astype(int).tolist()
    return sorted(set(start + i for i in local_idxs))

def extract_temporal_split_to_hdf5(seqs, cross_key, irrelevant_value, t_frames, out_path):
    sample_meta = []
    for i in range(len(seqs['pid'])):
        labels = seqs[cross_key][i]
        label = _unwrap_label(labels[-1] if isinstance(labels, list) else labels)
        if label == irrelevant_value:
            continue
        sample_meta.append({'seq_i': i, 'label': int(label)})

    n_samples = len(sample_meta)

    if os.path.exists(out_path):
        hf = h5py.File(out_path, 'a')
        completed = hf['completed'][:]
        print(f'Resuming {out_path}: {int(completed.sum())}/{n_samples} already done')
    else:
        hf = h5py.File(out_path, 'w')
        hf.create_dataset('images', shape=(n_samples, t_frames, IMG_SIZE, IMG_SIZE, 3), dtype='uint8')
        hf.create_dataset('labels', shape=(n_samples,), dtype='int64')
        hf.create_dataset('completed', shape=(n_samples,), dtype='bool')
        for idx, meta in enumerate(sample_meta):
            hf['labels'][idx] = meta['label']
        hf['completed'][:] = False
        hf.flush()
        completed = hf['completed'][:]
        print(f'Starting fresh: 0/{n_samples} for {out_path}')

    video_cache = {}
    n_done_now = 0
    pbar = tqdm(range(n_samples), desc=os.path.basename(out_path))
    for idx in pbar:
        if completed[idx]:
            continue
        meta = sample_meta[idx]
        i = meta['seq_i']
        image_refs = seqs['image'][i]
        bboxes = seqs['bbox'][i]
        n_available = len(image_refs)
        idxs = _sample_indices(n_available, t_frames, FPS, WINDOW_SECONDS)
        frames_out = np.zeros((t_frames, IMG_SIZE, IMG_SIZE, 3), dtype='uint8')
        for slot, frame_idx in enumerate(idxs):
            image_ref = image_refs[frame_idx]
            bbox = bboxes[frame_idx]
            video_path, frame_num = _resolve_video_path(image_ref)
            if video_path not in video_cache:
                video_cache[video_path] = cv2.VideoCapture(video_path)
            cap = video_cache[video_path]
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ok, frame = cap.read()
            if not ok:
                continue
            x1, y1, x2, y2 = [int(v) for v in bbox]
            crop = frame[max(y1,0):y2, max(x1,0):x2]
            if crop.size == 0:
                continue
            crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
            crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            frames_out[slot] = crop
        hf['images'][idx] = frames_out
        hf['completed'][idx] = True
        n_done_now += 1
        if n_done_now % 20 == 0:
            hf.flush()

    hf.flush()
    hf.close()
    for cap in video_cache.values():
        cap.release()
    print(f'Done: {n_samples} samples written to {out_path}')
    return n_samples

for split_name, seqs in [('train', train_seqs), ('val', val_seqs), ('test', test_seqs)]:
    out_path = f'{CROPS_DIR}/{split_name}_temporal_crops_v3.h5'
    extract_temporal_split_to_hdf5(seqs, CROSS_KEY, IRRELEVANT_VALUE, T_FRAMES, out_path)


Resuming /content/drive/MyDrive/PIE_thesis/crops/train_temporal_crops_v3.h5: 876/876 already done


train_temporal_crops_v3.h5:   0%|          | 0/876 [00:00<?, ?it/s]

Done: 876 samples written to /content/drive/MyDrive/PIE_thesis/crops/train_temporal_crops_v3.h5
Resuming /content/drive/MyDrive/PIE_thesis/crops/val_temporal_crops_v3.h5: 243/243 already done


val_temporal_crops_v3.h5:   0%|          | 0/243 [00:00<?, ?it/s]

Done: 243 samples written to /content/drive/MyDrive/PIE_thesis/crops/val_temporal_crops_v3.h5
Resuming /content/drive/MyDrive/PIE_thesis/crops/test_temporal_crops_v3.h5: 717/717 already done


test_temporal_crops_v3.h5:   0%|          | 0/717 [00:00<?, ?it/s]

Done: 717 samples written to /content/drive/MyDrive/PIE_thesis/crops/test_temporal_crops_v3.h5


## Next steps (once this finishes)

1. Reconnect with T4 GPU
2. Reuse Model B's training cells (Dataset class, class weights, model, train, evaluate) exactly as before, pointing at `*_temporal_crops_v3.h5` instead of the old files
3. Compare against: Model A 0.795, Model B v1 (frozen, wrong window) 0.787, Model B v2 (unfrozen, wrong window) 0.766

In [13]:
import torch
import torch.nn as nn
import numpy as np
import h5py
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
import pandas as pd
import random

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', device)

Device: cuda


In [24]:
IMG_SIZE = 224
normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

class PIETemporalDataset(Dataset):
    def __init__(self, h5_path, train=False):
        self.h5_path = h5_path
        self.train = train
        with h5py.File(h5_path, 'r') as hf:
            self.length = hf['images'].shape[0]
            self.t_frames = hf['images'].shape[1]
        self._hf = None

    def _ensure_open(self):
        if self._hf is None:
            self._hf = h5py.File(self.h5_path, 'r')

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        self._ensure_open()
        frames = self._hf['images'][idx]
        label = int(self._hf['labels'][idx])
        flip = self.train and random.random() < 0.5
        out = torch.zeros(self.t_frames, 3, IMG_SIZE, IMG_SIZE)
        for t in range(self.t_frames):
            img = torch.from_numpy(frames[t]).permute(2, 0, 1).float() / 255.0
            if flip:
                img = torch.flip(img, dims=[2])
            out[t] = normalize(img)
        return out, label

train_ds = PIETemporalDataset(f'{CROPS_DIR}/train_temporal_crops_v3.h5', train=True)
val_ds = PIETemporalDataset(f'{CROPS_DIR}/val_temporal_crops_v3.h5')
test_ds = PIETemporalDataset(f'{CROPS_DIR}/test_temporal_crops_v3.h5')
print('Dataset sizes:', len(train_ds), len(val_ds), len(test_ds))
print('T frames:', train_ds.t_frames)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)


Dataset sizes: 876 243 717
T frames: 16


In [6]:
with h5py.File(f'{CROPS_DIR}/train_temporal_crops_v3.h5', 'r') as hf:
    train_labels = hf['labels'][:]

unique, counts = np.unique(train_labels, return_counts=True)
print('Train class distribution:', dict(zip(unique.tolist(), counts.tolist())))
class_weights = torch.tensor([len(train_labels) / c for c in counts], dtype=torch.float32).to(device)
print('Class weights:', class_weights)

Train class distribution: {0: 623, 1: 253}
Class weights: tensor([1.4061, 3.4625], device='cuda:0')


In [14]:
backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT).features
backbone = backbone.to(device)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False

FEATURE_DIM = 1280
HIDDEN_DIM = 128

class TemporalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=FEATURE_DIM, hidden_size=HIDDEN_DIM, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.Linear(HIDDEN_DIM, 2)

    def forward(self, features):
        _, (h_n, _) = self.lstm(features)
        out = self.dropout(h_n[-1])
        return self.classifier(out)

def extract_features(frames_batch):
    b, t, c, h, w = frames_batch.shape
    flat = frames_batch.view(b * t, c, h, w)
    with torch.no_grad():
        feats = backbone(flat)
        feats = nn.functional.adaptive_avg_pool2d(feats, 1).view(b * t, -1)
    return feats.view(b, t, -1)

model = TemporalModel().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
print(model)

TemporalModel(
  (lstm): LSTM(1280, 128, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [15]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []
    for frames, y in loader:
        frames, y = frames.to(device), y.to(device)
        feats = extract_features(frames)
        if train:
            optimizer.zero_grad()
        out = model(feats)
        loss = criterion(out, y)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * frames.size(0)
        n += frames.size(0)
        preds = out.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return total_loss / max(n, 1), f1

In [27]:
EPOCHS = 20
PATIENCE = 8
history = []
best_val_f1 = 0.0
patience_counter = 0
for epoch in range(EPOCHS):
    train_loss, train_f1 = run_epoch(train_loader, train=True)
    val_loss, val_f1 = run_epoch(val_loader, train=False)
    print(f'Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f} train_f1={train_f1:.4f}  val_loss={val_loss:.4f} val_f1={val_f1:.4f}')
    history.append({'epoch': epoch+1, 'train_loss': train_loss, 'train_f1': train_f1, 'val_loss': val_loss, 'val_f1': val_f1})
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), f'{CHECKPOINTS_DIR}/model_b_v3_fixedwindow.pt')
        print(f'  New best val_f1={val_f1:.4f}, checkpoint saved')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}, best val_f1={best_val_f1:.4f}')
            break

print('Training done, best val_f1:', best_val_f1)

Epoch 1/20  train_loss=0.1980 train_f1=0.8821  val_loss=0.4291 val_f1=0.6891
  New best val_f1=0.6891, checkpoint saved
Epoch 2/20  train_loss=0.1400 train_f1=0.9349  val_loss=0.4393 val_f1=0.6981
  New best val_f1=0.6981, checkpoint saved
Epoch 3/20  train_loss=0.0864 train_f1=0.9538  val_loss=0.5457 val_f1=0.7170
  New best val_f1=0.7170, checkpoint saved
Epoch 4/20  train_loss=0.0668 train_f1=0.9706  val_loss=0.6064 val_f1=0.7048
Epoch 5/20  train_loss=0.0483 train_f1=0.9765  val_loss=0.6259 val_f1=0.7193
  New best val_f1=0.7193, checkpoint saved
Epoch 6/20  train_loss=0.0379 train_f1=0.9843  val_loss=0.7557 val_f1=0.6726
Epoch 7/20  train_loss=0.0266 train_f1=0.9862  val_loss=0.7075 val_f1=0.7327
  New best val_f1=0.7327, checkpoint saved
Epoch 8/20  train_loss=0.0430 train_f1=0.9783  val_loss=0.6726 val_f1=0.7059
Epoch 9/20  train_loss=0.0190 train_f1=0.9941  val_loss=0.5960 val_f1=0.7257
Epoch 10/20  train_loss=0.0088 train_f1=0.9980  val_loss=0.7997 val_f1=0.6869
Epoch 11/20  t

In [28]:
model.load_state_dict(torch.load(f'{CHECKPOINTS_DIR}/model_b_v3_fixedwindow.pt'))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for frames, y in test_loader:
        frames = frames.to(device)
        feats = extract_features(frames)
        out = model(feats)
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

f1 = f1_score(all_labels, all_preds, zero_division=0)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
print({'model': 'Model B v3 - Fixed window (last 2.0s)', 'f1': f1, 'precision': precision, 'recall': recall})
print(classification_report(all_labels, all_preds, target_names=['not-crossing', 'crossing']))

{'model': 'Model B v3 - Fixed window (last 2.0s)', 'f1': 0.784503631961259, 'precision': 0.7788461538461539, 'recall': 0.7902439024390244}
              precision    recall  f1-score   support

not-crossing       0.92      0.91      0.91       512
    crossing       0.78      0.79      0.78       205

    accuracy                           0.88       717
   macro avg       0.85      0.85      0.85       717
weighted avg       0.88      0.88      0.88       717

